In [1]:
import os
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import mllabs  # ml-labs: our experiment management framework

import sys
print(sys.version)

for i in [pd, pl, np, plt, sns, scipy, sklearn, tf, xgb, lgb, cb, mllabs]:
    if hasattr(i, '__version__'):
        print(i.__name__, i.__version__)
    else:
        print(i.__name__)

from IPython.display import Markdown

from mllabs import Connector, Experimenter, ColSelector
from mllabs.collector import MetricCollector, ModelAttrCollector, StackingCollector
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

data_path = Path('data')

# Binary-encode several categorical features using Polars expressions.
# We also derive indicator columns for internet-service-related features
# ('No internet service' is treated as a separate binary flag).
dict_expr = {
    'gender': (pl.col('gender') == 'Male').cast(pl.Int8),
    'No_Internet': (pl.col('DeviceProtection') == 'No internet service').cast(pl.Int8),
    'DSL_Y': (pl.col('InternetService') == 'DSL').cast(pl.Int8)
}
for i in ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService']:
    dict_expr[i] = (pl.col(i) == 'Yes').cast(pl.Int8)

for i in ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'MultipleLines']:
    dict_expr[i + '_Y'] = (pl.col(i) == 'Yes').cast(pl.Int8)

# PolarsLoader reads CSVs into Polars DataFrames.
# ExprProcessor applies the expression dict above in a single vectorized pass.
# PandasConverter converts to pandas, setting 'id' as the index.
loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr),
    PandasConverter(index_col='id')
)

df_train = loader.fit_transform([data_path / 'train.csv']).assign(
    Churn=lambda x: (x['Churn'] == 'Yes').astype('int8')
)
df_test = loader.transform([data_path / 'test.csv'])

# Variable groupings
# X_bin  : original binary features (Yes/No encoded as 0/1)
# X_tri  : 3-level categorical features (Yes / No / No internet service)
# X_bin2 : derived binary flags from X_tri (e.g. OnlineSecurity_Y)
# X_num  : numerical features
# X_nom  : nominal categoricals kept as strings for tree-based models
X_bin = ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService', 'gender', 'SeniorCitizen']
X_tri = ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport']
X_bin2 = ['{}_Y'.format(i) for i in X_tri] + ['No_Internet', 'DSL_Y', 'MultipleLines_Y']
X_tri.append('InternetService')
X_tri.append('MultipleLines')
X_num = ['TotalCharges', 'MonthlyCharges', 'tenure']
X_nom = ['PaymentMethod', 'Contract']
X_all = X_bin + X_tri + X_bin2 + X_num + X_nom

target = 'Churn'

2026-03-11 00:54:44.375918: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-11 00:54:44.412929: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-11 00:54:45.432706: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sun9sun9/python312/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: Fu

3.12.12 (main, Dec 27 2025, 11:08:36) [GCC 13.3.0]
pandas 2.3.3
polars 1.38.1
numpy 2.3.5
matplotlib.pyplot
seaborn 0.13.2
scipy 1.16.3
sklearn 1.8.0
tensorflow 2.20.0
xgboost 3.2.0
lightgbm
catboost 1.2.8
mllabs 0.6.1


In [2]:
if os.path.exists('exp/skf5'):
    e_skf5 = Experimenter.load('exp/skf5', df_train)
    if e_skf5.status == 'closed':
        e_skf5.reopen_exp()
else:
    e_skf5 = Experimenter.create(
        df_train, 'exp/skf5', title='The experimentation using 5-fold StratifiedKFold',
        sp=StratifiedKFold(n_splits=5, shuffle=True, random_state=1),
        splitter_params={'y': target}
    )

e_skf5.set_grp('pre', role='stage', method='transform')
y_edges = {'y': [(None, target)]}
e_skf5.set_grp(
    'clf', role='head', method='predict_proba', edges=y_edges
)
e_skf5.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges=y_edges), slice(-1, None), roc_auc_score, include_train = True
    )
)
e_skf5.add_collector(
    StackingCollector(
        'stacking', Connector(edges=y_edges),
        slice(-1, None), method='mean', experimenter = e_skf5
    )
)
e_skf5.set_grp('xgb', parent='clf', processor=xgb.XGBClassifier,
    adapter=XGBoostAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 
        'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
        'learning_rate': 0.05, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 10
    }
)

e_skf5.set_grp('lgb', parent='clf', processor=lgb.LGBMClassifier,
    adapter=LightGBMAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)

e_skf5.set_grp('cb', parent='clf', processor=cb.CatBoostClassifier,
    adapter=CatBoostAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': 0, 'random_state': 1, 'max_depth': 5, 'colsample_bylevel': 0.75
    }
)
Markdown(
    e_skf5.desc_spec()
)


Loaded: 5 node(s), 4 group(s), 5 fold(s)


## The experimentation using 5-fold StratifiedKFold

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=5, random_state=1, shuffle=True)` |
| **Inner Splitter (sp_v)** | None |
| **Splitter Params** | `{y='Churn'}` |
| **Outer Folds** | 5 |
| **Inner Folds** | 1 |

In [3]:
params = [(3, 2500, 0.05), (4, 1400, 0.05), (5, 2000, 0.025)]
for i, (max_depth, n_estimators, learning_rate) in enumerate(params):
    e_skf5.set_node(
        'xgb_{}'.format(i), grp = 'xgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
        params={'max_depth': max_depth, 'n_estimators': n_estimators, 'learning_rate': learning_rate}
    )
e_skf5.exp()

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


In [4]:
e_skf5.get_collector('AUC').get_metrics_agg()[0]

,valid,train_sub
xgb_0,0.916618,0.918617
xgb_1,0.916575,0.919440
xgb_2,0.916595,0.920708


In [25]:
params = [(7, 2500, 0.05), (15, 2100, 0.025)]
for i, (nl, n_estimators, learning_rate) in enumerate(params):
    e_skf5.set_node(
        'lgb_{}'.format(i), grp = 'lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
        params={'num_leaves': nl, 'n_estimators': n_estimators, 'learning_rate': learning_rate}
    )
e_skf5.exp()

Experimenting 2 node(s)
Exp 5/5 (100%)                                    
Experimentation complete: 2 node(s)


In [28]:
e_skf5.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False)

,valid,train_sub
xgb_0,0.916618,0.918617
xgb_2,0.916595,0.920708
xgb_1,0.916575,0.919440
lgb_0,0.916505,0.918883
lgb_1,0.916439,0.919267


# Stacking

In [29]:
df_stacking = e_skf5.collectors['stacking'].get_dataset().sort_index()
df_stacking.head()

,xgb_0__Churn_1,xgb_1__Churn_1,lgb_0__Churn_1,xgb_2__Churn_1,lgb_1__Churn_1,Churn
id,,,,,,
0,0.009932,0.009954,0.009811,0.010675,0.009994,0
1,0.000652,0.000807,0.000936,0.000784,0.001085,0
2,0.249691,0.246377,0.252500,0.244610,0.242516,0
3,0.790442,0.778810,0.816275,0.772328,0.805303,1
4,0.799588,0.789883,0.810742,0.789813,0.801976,1


In [30]:
if os.path.exists('exp/stacking'):
    e_stacking = Experimenter.load('exp/stacking', df_stacking)
    if e_stacking.status == 'closed':
        e_stacking.reopen_exp()
else:
    e_stacking = Experimenter.create(
        df_stacking, 'exp/stacking', title='Stacking',
        sp=StratifiedKFold(n_splits=5, random_state=1, shuffle=True),
        splitter_params=y_edges
    )

e_stacking.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges={'y': [(None, target)]}),
        slice(-1, None),
        roc_auc_score,
        include_train = True
    )
)

e_stacking.set_grp('pre', role='stage', method='transform')
e_stacking.build()
e_stacking.set_grp(
    'clf', role='head', method='predict_proba',
    edges={'y': [(None, target)]}
)
e_stacking.set_grp('lr', parent='clf', processor = LogisticRegression)

Loaded: 3 node(s), 3 group(s), 5 fold(s)
Building 0 node(s)
Build 5/5 (100%)        
Build complete: 0 node(s)


{'result': 'skip',
 'grp': <mllabs._pipeline.PipelineGroup at 0x7a45cc514320>,
 'affected_nodes': []}

In [41]:
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_1', 'xgb_2']]
e_stacking.set_node('lr1', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_2']]
e_stacking.set_node('lr2', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_1', 'xgb_2']]
e_stacking.set_node('lr3', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_2', 'lgb_0']]
e_stacking.set_node('lr4', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_1', 'xgb_2', 'lgb_0', 'lgb_1']]
e_stacking.set_node('lr5', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_1', 'lgb_0']]
e_stacking.set_node('lr6', grp = 'lr', edges = {'X': [(None, X_sel)]})
e_stacking.exp()

Experimenting 1 node(s)
Exp 5/5 (100%)                
Experimentation complete: 1 node(s)


In [42]:
e_stacking.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending=False)

,valid,train_sub,valid_sub
lr4,0.916719,0.916582,0.917908
lr5,0.916674,0.916533,0.917898
lr2,0.916671,0.916535,0.917849
lr6,0.916666,0.916526,0.917883
lr1,0.916657,0.916520,0.917851
lr3,0.916620,0.916481,0.917832


In [9]:
e_skf5.trainers['trainer']

# Submission

In [38]:
e_skf5.add_trainer('trainer')
e_skf5.trainers['trainer'].select_head(['xgb_0', 'xgb_2', 'lgb_0'])
e_skf5.trainers['trainer'].train()

lgb_0 1/1 (100%)                                  
Train complete: 1 node(s)


In [43]:
e_stacking.add_trainer('trainer')
e_stacking.trainers['trainer'].select_head(['lr4'])
e_stacking.trainers['trainer'].train()

lr4 1/1 (100%)                 
Train complete: 1 node(s)


In [77]:
df_result = e_stacking.trainers['trainer'].to_inferencer(slice(-1, None)).process(
    e_skf5.trainers['trainer'].to_inferencer(slice(-1, None)).process(df_test)
)
df_result.iloc[:, -1].rename('Churn')

id
594194    0.056085
594195    0.035684
594196    0.066604
594197    0.036301
594198    0.502649
            ...   
848844    0.035699
848845    0.858890
848846    0.204861
848847    0.035977
848848    0.350693
Name: Churn, Length: 254655, dtype: float64

In [78]:
df_result.iloc[:, -1].rename('Churn').to_csv('data/submission.csv')

In [79]:
# !kaggle competitions submit -c playground-series-s6e3 -f data/submission.csv -m "3"

100%|██████████████████████████████████████| 6.51M/6.51M [00:03<00:00, 1.89MB/s]
Successfully submitted to Predict Customer Churn